## Azure AI Foundry SDK Hands-on
This notebook demonstrates how to use the Azure AI Foundry SDK for AI applications, providing enhanced capabilities over traditional Azure OpenAI integration.

## Overview
This hands-on session helps you understand Azure AI Foundry SDK integration and text analysis capabilities.

The Azure AI Foundry SDK provides a unified interface for working with Azure AI services, offering improved connection management, monitoring, and collaboration features compared to direct OpenAI API usage.

For more information about Azure AI Foundry, refer to the [official documentation](https://learn.microsoft.com/azure/ai-studio/)

#### Prerequisite
Please complete the **00_Setup.ipynb** notebook before running this notebook.

## Table of Contents

[Overview](#overview)  
[Getting started with Azure AI Foundry SDK](#getting-started-with-azure-ai-foundry-sdk)  
[Build your first prompt with Foundry](#build-your-first-prompt-with-foundry)  

[Use Cases](#use-cases)  
[1. Summarize Text](#summarize-text)  
[2. Classify Text](#classify-text)  
[3. Generate New Product Names](#generate-new-product-names)  
[4. Embeddings with Foundry](#embeddings-with-foundry)  
[5. Model Performance Monitoring](#model-performance-monitoring)  

[References](#references)

### Build your first prompt with Azure AI Foundry
This exercise provides a basic introduction for submitting prompts using the Azure AI Foundry SDK.

**Steps you will complete**:

1. Install Azure AI Foundry SDK and dependencies
2. Load standard helper libraries and establish Foundry connection
3. Connect to your deployed models through Foundry
4. Create a simple prompt for the model
5. Submit your request using the Foundry SDK
6. Monitor and track your API usage

#### Azure Authentication - see setup notebook for explanation. You may need to rerun this if the credetials expire

In [ ]:
# Azure Authentication using Helper Module
import os
from dotenv import load_dotenv
from azure_auth_helper import authenticate_azure

# Load environment variables
load_dotenv("./.env")

# Get tenant ID from environment variables
TENANT_ID = os.getenv('AZURE_TENANT_ID')
if not TENANT_ID:
    raise ValueError("AZURE_TENANT_ID not found in .env file. Please add it to your .env file.")

print(f"🏢 Using tenant ID: {TENANT_ID}")

# Authenticate with Azure using browser authentication (interactive)
# Opens browser window for Azure login with specific tenant
credential = authenticate_azure(
    auth_method='browser', 
    tenant_id=TENANT_ID
)

# Test that the credential actually works
print(f"🔍 Testing credential type: {type(credential).__name__}")
try:
    # Try to get a token to validate the credential
    token = credential.get_token("https://management.azure.com/.default")
    print("✅ Credential test successful!")
    print(f"Token expires: {token.expires_on}")
except Exception as e:
    print(f"❌ Credential test failed: {e}")
    print(f"   Error type: {type(e).__name__}")
    raise

print("🎉 Ready to use Azure AI Foundry!")

🏢 Using tenant ID: 7ec824c6-48d9-4e32-b7a1-9a3df8bdcf66


DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: Authentication failed: AADSTS700016: Application with identifier 'ffbf62a3-8d06-4708-97e6-4d5a00cb1cc3' was not found in the directory 'Contoso'. This can happen if the application has not been installed by the administrator of the tenant or consented to by any user in the tenant. You may have sent your authentication request to the wrong tenant. Trace ID: e96f5c14-9403-4b13-b2c3-bfd1053e3000 Correlation ID: 569291f2-b865-42c1-87c8-defba96b8d96 Timestamp: 2025-08-26 23:42:57Z
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/python/identity/defaultazurecredential/troubleshoot.


ℹ️ No existing credentials found, will authenticate...
🌐 Starting interactive browser authentication...
🏢 Authenticating with tenant: 7ec824c6-48d9-4e32-b7a1-9a3df8bdcf66


### 2. Import helper libraries and establish Foundry connection

In [18]:
import os
import numpy as np
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from datetime import datetime

# Load environment variables
load_dotenv("./.env")

project = AIProjectClient(
    endpoint=os.getenv("FOUNDRY_API_ENDPOINT"),
    credential=credential,
)

print("✅ Successfully connected to Azure AI Foundry")
print(f"📊 Project endpoint: {os.getenv('FOUNDRY_API_ENDPOINT')}")
print(f"🏢 Resource Group: {os.getenv('AZURE_RESOURCE_GROUP')}")
print(f"🔑 Using DefaultAzureCredential for project authentication")

✅ Successfully connected to Azure AI Foundry
📊 Project endpoint: https://hannahhowell-3776-resource.services.ai.azure.com/api/projects/hannahhowell-3776
🏢 Resource Group: rg-hannahhowell-3776
🔑 Using DefaultAzureCredential for project authentication


### 3. Get deployed model

In [19]:
# Get model deployment names from environment
gpt4o_model = os.getenv('GPT4O_DEPLOYMENT_NAME', 'gpt-4o')

## 4. Prompt Design  

"The magic of large language models is that by being trained to minimize this prediction error over vast quantities of text, the models end up learning concepts useful for these predictions. For example, they learn concepts like"(1):

* how to spell
* how grammar works
* how to paraphrase
* how to answer questions
* how to hold a conversation
* how to write in many languages
* how to code
* etc.

#### How to control a large language model  
"Of all the inputs to a large language model, by far the most influential is the text prompt"

Large language models can be prompted to produce output in a few ways:

- Instruction: Tell the model what you want
- Completion: Induce the model to complete the beginning of what you want
- Demonstration: Show the model what you want, with either:
  - A few examples in the prompt
  - Many hundreds or thousands of examples in a fine-tuning training dataset

#### There are three basic guidelines to creating prompts:

**Show and tell**. Make it clear what you want either through instructions, examples, or a combination of the two. If you want the model to rank a list of items in alphabetical order or to classify a paragraph by sentiment, show it that's what you want.

**Provide quality data**. If you're trying to build a classifier or get the model to follow a pattern, make sure that there are enough examples. Be sure to proofread your examples — the model is usually smart enough to see through basic spelling mistakes and give you a response, but it also might assume this is intentional and it can affect the response.

**Check your settings.** The temperature and top_p settings control how deterministic the model is in generating a response. If you're asking it for a response where there's only one right answer, then you'd want to set these lower. If you're looking for more diverse responses, then you might want to set them higher. The number one mistake people make with these settings is assuming that they're "cleverness" or "creativity" controls.

Source: https://github.com/Azure/OpenAI/blob/main/How%20to/Completions.md

In [20]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

# Now we can use the Foundry project client to get the OpenAI client
# This gives us all the Foundry benefits like monitoring and tracking
models = project.get_openai_client(api_version="2024-10-21")

response = models.chat.completions.create(
    model=gpt4o_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": text_prompt}
    ]
)

response.choices[0].message.content

BadRequestError: Error code: 400 - {'error': {'code': 'Tenant provided in token does not match resource token', 'message': 'Token tenant 72f988bf-86f1-41af-91ab-2d7cd011db47 does not match resource tenant.'}}

### Repeat the same call, how do the results compare?

In [ ]:
# Repeat the same call to see variation in responses
response = models.chat.completions.create(
    model=gpt4o_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": text_prompt}
    ]
)

response.choices[0].message.content

### 5. Submit!

## Use Cases

### 1. Summarize Text
Let's use Azure AI Foundry to summarize a piece of text with enhanced monitoring.

In [ ]:
# Text to summarize
text_to_summarize = """
Azure AI Foundry is Microsoft's comprehensive platform for building, deploying, and managing AI applications. 
It provides a unified experience for data scientists, developers, and business users to collaborate on AI projects. 
The platform includes capabilities for model development, deployment, monitoring, and governance. 
With Azure AI Foundry, organizations can accelerate their AI journey while maintaining security, compliance, and ethical AI practices. 
The platform supports various AI workloads including natural language processing, computer vision, and machine learning.
"""

summarize_prompt = f"Please summarize the following text in 2-3 sentences:\n\n{text_to_summarize}"

# Get summary with tracking
response, metrics = enhanced_completion(summarize_prompt, temperature=0.3)

if response:
    summary = response.choices[0].message.content
    print("📝 **Summary:**")
    print(summary)
    print(f"\n📊 **Metrics:**")
    print(f"   • Total tokens used: {metrics['total_tokens']}")
    print(f"   • Prompt tokens: {metrics['prompt_tokens']}")
    print(f"   • Completion tokens: {metrics['completion_tokens']}")

### 2. Classify Text
Demonstrate text classification using Azure AI Foundry with enhanced error handling.

In [ ]:
# Text classification example
texts_to_classify = [
    "I love this new smartphone! The camera quality is amazing.",
    "The delivery was delayed and the package was damaged.",
    "The weather forecast shows rain for the next three days."
]

classify_prompt = """
Classify the following text into one of these categories: POSITIVE, NEGATIVE, NEUTRAL.
Provide only the category name.

Text: {text}
Category:
"""

print("🏷️ **Text Classification Results:**")
print()

all_metrics = []
for i, text in enumerate(texts_to_classify, 1):
    prompt = classify_prompt.format(text=text)
    response, metrics = enhanced_completion(prompt, temperature=0.1, max_tokens=10)
    
    if response:
        classification = response.choices[0].message.content.strip()
        print(f"**Text {i}:** {text}")
        print(f"**Classification:** {classification}")
        print(f"**Tokens used:** {metrics['total_tokens']}")
        print()
        all_metrics.append(metrics)

# Calculate average token usage
if all_metrics:
    avg_tokens = sum(m['total_tokens'] for m in all_metrics) / len(all_metrics)
    print(f"📈 **Average tokens per classification:** {avg_tokens:.1f}")

### 3. Generate New Product Names
Use Azure AI Foundry for creative content generation with multiple variations.

In [ ]:
# Product name generation
product_description = "An eco-friendly water bottle made from recycled materials with temperature control features"

generation_prompt = f"""
Generate 5 creative and catchy product names for the following product:

Product: {product_description}

Requirements:
- Names should be memorable and marketable
- Reflect the eco-friendly and innovative nature
- Keep names under 20 characters

Product Names:
"""

response, metrics = enhanced_completion(generation_prompt, temperature=0.8, max_tokens=200)

if response:
    product_names = response.choices[0].message.content
    print("🚀 **Generated Product Names:**")
    print(product_names)
    print(f"\n💡 **Generation Metrics:**")
    print(f"   • Creative temperature: 0.8")
    print(f"   • Tokens used: {metrics['total_tokens']}")
    print(f"   • Generation efficiency: {metrics['completion_tokens']/metrics['total_tokens']:.2%}")

### 4. Embeddings with Azure AI Foundry
Demonstrate text embeddings generation with enhanced error handling and monitoring.

In [ ]:
# Text embedding example
def get_embeddings_with_tracking(text, model=None):
    """Get embeddings with automatic tracking using Azure AI Foundry"""
    if model is None:
        model = os.getenv('EMBEDDINGS_DEPLOYMENT_NAME', 'text-embedding-ada-002')
    
    try:
        # Use the project client to get embeddings
        embeddings_client = project.inference.get_embeddings_client()
        response = embeddings_client.embed(
            model=model,
            input=text
        )
        
        embedding = response.data[0].embedding
        metrics = {
            "timestamp": datetime.now().isoformat(),
            "model": model,
            "input_length": len(text),
            "embedding_dimensions": len(embedding),
            "total_tokens": response.usage.total_tokens
        }
        
        return embedding, metrics
    except Exception as e:
        print(f"❌ Embedding error: {e}")
        return None, None

# Sample texts for embeddings
sample_texts = [
    "Azure AI Foundry simplifies AI development",
    "Machine learning requires quality data",
    "Natural language processing enables chatbots"
]

print("🔢 **Text Embeddings with Azure AI Foundry:**")
print()

embeddings_data = []
for i, text in enumerate(sample_texts, 1):
    embedding, metrics = get_embeddings_with_tracking(text)
    
    if embedding:
        embeddings_data.append({
            "text": text,
            "embedding": embedding,
            "metrics": metrics
        })
        
        print(f"**Text {i}:** {text}")
        print(f"**Embedding dimensions:** {metrics['embedding_dimensions']}")
        print(f"**Tokens used:** {metrics['total_tokens']}")
        print(f"**First 5 dimensions:** {embedding[:5]}")
        print()

# Calculate similarity between first two embeddings
if len(embeddings_data) >= 2:
    emb1 = np.array(embeddings_data[0]["embedding"])
    emb2 = np.array(embeddings_data[1]["embedding"])
    
    # Cosine similarity
    similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    print(f"🔗 **Cosine similarity between first two texts:** {similarity:.4f}")

### 5. Model Performance Monitoring
Azure AI Foundry provides enhanced monitoring capabilities. Let's demonstrate session tracking.

In [ ]:
# Session monitoring and analytics
class FoundrySessionTracker:
    def __init__(self):
        self.session_metrics = []
        self.session_start = datetime.now()
    
    def add_metric(self, metric_data):
        self.session_metrics.append(metric_data)
    
    def get_session_summary(self):
        if not self.session_metrics:
            return "No metrics recorded"
        
        total_tokens = sum(m.get('total_tokens', 0) for m in self.session_metrics)
        total_requests = len(self.session_metrics)
        avg_tokens = total_tokens / total_requests if total_requests > 0 else 0
        
        session_duration = datetime.now() - self.session_start
        
        summary = f"""
🎯 **Azure AI Foundry Session Summary**
   • Session duration: {session_duration.total_seconds():.1f} seconds
   • Total API requests: {total_requests}
   • Total tokens consumed: {total_tokens:,}
   • Average tokens per request: {avg_tokens:.1f}
   • Models used: {len(set(m.get('model', 'unknown') for m in self.session_metrics))}
        """
        
        return summary

# Initialize session tracker
session_tracker = FoundrySessionTracker()

# Add previous metrics to tracker (simulating session tracking)
sample_metrics = [
    {"model": selected_model, "total_tokens": 150, "timestamp": datetime.now().isoformat()},
    {"model": selected_model, "total_tokens": 75, "timestamp": datetime.now().isoformat()},
    {"model": embeddings_model, "total_tokens": 25, "timestamp": datetime.now().isoformat()}
]

for metric in sample_metrics:
    session_tracker.add_metric(metric)

print(session_tracker.get_session_summary())

## Key Benefits of Azure AI Foundry SDK

This notebook demonstrated several advantages of using Azure AI Foundry:

### 🚀 **Enhanced Features**
- **Unified Connection Management**: Single authentication for all AI services
- **Automatic Monitoring**: Built-in token tracking and performance metrics
- **Error Handling**: Improved retry logic and error reporting
- **Collaboration**: Team-based development with shared resources

### 📊 **Monitoring & Analytics**
- Real-time token usage tracking
- Performance metrics collection
- Session-based analytics
- Cost optimization insights

### 🔒 **Security & Governance**
- Centralized security policies
- Audit trails for all API calls
- Compliance monitoring
- Resource access control

### 🛠️ **Developer Experience**
- Simplified SDK with enhanced capabilities
- Better debugging and logging
- Integrated development environment
- Version control for prompts and models

## References

- [Azure AI Foundry Documentation](https://learn.microsoft.com/azure/ai-studio/)
- [Azure AI Foundry SDK Reference](https://learn.microsoft.com/python/api/overview/azure/ai-ml-readme)
- [Azure OpenAI Service](https://learn.microsoft.com/azure/ai-services/openai/)
- [Prompt Engineering Guide](https://learn.microsoft.com/azure/ai-services/openai/concepts/prompt-engineering)
- [Azure AI Safety and Responsible AI](https://learn.microsoft.com/azure/ai-services/responsible-use-of-ai-overview)